In [9]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

### 1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.

In [10]:
train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')

### 2. Проведите предобработку: 
• Приведите тексты к нижнему регистру. 

• Замените все, кроме букв и цифр, на пробелы — это облегчит дальнейшее разделение текста на слова. Для такой замены в строке text подходит следующий вызов: re.sub(’[ ^ a−zA−Z0−9]’,’␣’,text.lower())

In [11]:
def clean_text(text):
    return re.sub('[^a-zA-Z0-9]', ' ', str(text).lower())

train['FullDescription'] = train['FullDescription'].apply(clean_text)
test['FullDescription'] = test['FullDescription'].apply(clean_text)

• Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только те слова, которые встречаются хотя бы в 5 объектах (параметр min_df у TfidfVectorizer).

In [12]:
vectorizer = TfidfVectorizer(min_df=5)
X_text_train = vectorizer.fit_transform(train['FullDescription'])
X_text_test = vectorizer.transform(test['FullDescription'])

• Замените пропуски в столбцах LocationNormalized и ContractTime на специальную строку ’nan’. Код для этого был приведен выше. 

• Примените DictVectorizer для получения one-hot-кодирования признаков LocationNormalized и ContractTime.

In [13]:
for col in ['LocationNormalized', 'ContractTime']:
    train[col] = train[col].fillna('nan')
    test[col] = test[col].fillna('nan')
    
dict_vec = DictVectorizer(sparse=True)
X_cat_train = dict_vec.fit_transform(train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_cat_test = dict_vec.transform(test[['LocationNormalized', 'ContractTime']].to_dict('records'))

• Объедините все полученные признаки в одну матрицу "объекты-признаки". Обратите внимание, что матрицы для текстов и категориальных признаков являются разреженными. Для объединения их столбцов нужно воспользоваться функцией scipy.sparse.hstack.

In [14]:
X_train = hstack([X_text_train, X_cat_train])
X_test = hstack([X_text_test, X_cat_test])
y_train = train['SalaryNormalized']

### 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.

In [15]:
model = Ridge(alpha=1)
model.fit(X_train, y_train)
preds = model.predict(X_test)

### 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание. Укажите их через пробел.

In [16]:
ans = f"{preds[0]:.2f} {preds[1]:.2f}"
print(f"Ответ: {ans}")

Ответ: 56575.32 37197.40
